# Datos diarios: validación fuera de muestra con 15+ años

**Autor:** Samuel  
**Institución:** Tecnológico de Monterrey  
**Área:** Pairs trading en Forex  
**Notebooks previos:** Backtest-Corregido, Beta-Anclado-OU, Remuestreo-Costos, Amplitud, Eventos

Los resultados horarios dependen de ~2 años de datos: al refrescarlos, el bruto cambió ~40%. Aquí repetimos el procedimiento de *Amplitud* con datos **diarios desde 2010**, y separamos la muestra en dos bloques: las decisiones de diseño se toman con el primero y el segundo se evalúa **una sola vez**, sin ajustar nada después.

## Estructura de este Notebook
1. Fase 0: Datos diarios y corte en dos bloques
2. Fase 1: Motor (adaptado a barras diarias) y parámetros
3. Fase 2: Selección de canastas y operación en el bloque de diseño
4. Fase 3: Placebo del proceso completo
5. Fase 4: Evaluación única en el bloque de prueba
6. Conclusión

## Fase 0:

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings
from itertools import combinations
from scipy import stats
warnings.filterwarnings('ignore')
import yfinance as yf


In [ ]:
tickers = {
    'Baht_TH': 'USDTHB=X',  'Euro': 'EURUSD=X',     'Shekel_IL': 'USDILS=X', 'Won_KR': 'USDKRW=X',
    'Dolar_NZ': 'NZDUSD=X', 'Dolar_AUS': 'AUDUSD=X', 'Corona_CZ': 'USDCZK=X', 'Corona_SE': 'USDSEK=X',
    'Franco_SUI': 'USDCHF=X', 'Forinto_HU': 'USDHUF=X', 'Dolar_SG': 'USDSGD=X', 'Yen': 'USDJPY=X',
    'Dolar_CAN': 'USDCAD=X', 'Libra': 'GBPUSD=X',    'Peso_MX': 'USDMXN=X',  'Corona_NO': 'USDNOK=X',
    'Zloty_PL': 'USDPLN=X', 'Rand_ZA': 'USDZAR=X',
}
raw_data = yf.download(list(tickers.values()), start='2010-01-01', interval='1d',
                       progress=False, auto_adjust=True)['Close']
print(raw_data.shape)

In [ ]:
t2n = {v: k for k, v in tickers.items()}
raw = raw_data.rename(columns=t2n)[list(tickers)]
if raw.index.tz is not None:
    raw.index = raw.index.tz_localize(None)
raw.index = raw.index.normalize()
raw = raw[~raw.index.duplicated()].sort_index()

cob = pd.DataFrame({'primer_dato': raw.apply(lambda s: s.first_valid_index()),
                    'faltantes_%': (raw.isna().mean() * 100).round(2)})
display(cob.sort_values('primer_dato'))

INICIO_MAX = pd.Timestamp('2012-01-01')           # exigimos historia desde, a más tardar, 2012
aceptadas = [c for c in raw.columns if cob.loc[c, 'primer_dato'] <= INICIO_MAX and cob.loc[c, 'faltantes_%'] < 3]
print("Descartadas:", [c for c in raw.columns if c not in aceptadas])

data = raw[aceptadas].ffill(limit=3).dropna()     # ffill solo hasta 3 días (festivos); el resto se descarta
print(data.shape, data.index.min().date(), '->', data.index.max().date())